In [ ]:
import os
import numpy as np
from nilearn import image
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns

In [ ]:
data_dir = "/Users/silvycollin/Documents/research/narrativeProject/21styear_story/data_masks/TPJ/"
foundationScores_dir = "/Users/silvycollin/Documents/GitHub/wholeBrain_narratives_MAC_MFT/text/foundationScores/"

In [ ]:
# ------------------------
# load mask data
# ------------------------
left_TPJ = np.load(data_dir + "left_TPJ_3D_clean.npy")
right_TPJ = np.load(data_dir + "right_TPJ_3D_clean.npy")

# ------------------------
# load foundation scores 
# ------------------------
#start and end column is start and end of the sentences in seconds, TR was 1.5 sec
foundationScores = pd.read_excel(foundationScores_dir + "21styear_MFT_MAC.xlsx") 


In [ ]:
# foundationScoresSelect = foundationScores[["MAC_a_fairness_virtue","MAC_a_family_virtue","MAC_a_reciprocity_virtue",
#                                            "MAC_a_deference_virtue","MAC_a_property_virtue","MAC_a_group_virtue",
#                                            "MAC_a_property_virtue",
#                                            "MAC_a_fairness_vice","MAC_a_family_vice","MAC_a_reciprocity_vice",
#                                            "MAC_a_deference_vice","MAC_a_property_vice","MAC_a_group_vice",
#                                            "MAC_a_property_vice",
#                                            "MAC_a_moral_nonmoral","start","end"]]
foundationScoresSelect = foundationScores[["MAC_a_moral_nonmoral","start","end"]]
foundationScoresSelect = foundationScoresSelect.copy()

# go from seconds to TRs for start and end column
foundationScoresSelect["startTR"] = foundationScoresSelect["start"] / 1.5
foundationScoresSelect["endTR"] = foundationScoresSelect["end"] / 1.5

# round to full TRs/volumes
foundationScoresSelect["startTR"] = foundationScoresSelect["startTR"].round(0).astype(int)
foundationScoresSelect["endTR"] = foundationScoresSelect["endTR"].round(0).astype(int)

# shift by 3 TRs for the HRF
foundationScoresSelect['startTR'] = foundationScoresSelect['startTR'] + 3
foundationScoresSelect['endTR'] = foundationScoresSelect['endTR'] + 3


In [ ]:
# average across all voxels in the mask
left_TPJ_mean = left_TPJ.mean(axis=0)
print("check if averaging correctly, shape of output left_TPJ: ", left_TPJ_mean.shape)

right_TPJ_mean = right_TPJ.mean(axis=0)
print("check if averaging correctly, shape of output right_TPJ: ", right_TPJ_mean.shape)

# determine number of subjects
n_subjects = left_TPJ_mean.shape[1]

# pre-create the necessary columns
for s in range(n_subjects):
    foundationScoresSelect[f"left_TPJ_subj{s+1}"] = np.nan
    foundationScoresSelect[f"right_TPJ_subj{s+1}"] = np.nan

for index, row in foundationScoresSelect.iterrows():

    valueStart = int(row["startTR"])
    valueEnd = int(row["endTR"])

    for currSubj in range(n_subjects):

        # left TPJ
        col_name_leftTPJ = f"left_TPJ_subj{currSubj+1}"
        foundationScoresSelect.loc[index, col_name_leftTPJ] = left_TPJ_mean[valueStart:valueEnd+1, currSubj].mean()

        # right TPJ
        col_name_rightTPJ = f"right_TPJ_subj{currSubj+1}"
        foundationScoresSelect.loc[index, col_name_rightTPJ] = right_TPJ_mean[valueStart:valueEnd+1, currSubj].mean()

In [ ]:
# select only the moral sentences
foundationScoresSelect_nonzero = foundationScoresSelect[foundationScoresSelect['MAC_a_moral_nonmoral'] != 0]


In [ ]:
foundationScoresSelect_nonzero = foundationScoresSelect_nonzero.drop(columns=['start', 'end', 'startTR', 'endTR'])


In [ ]:
# split in left TPJ and right TPJ
foundationScoresSelect_nonzero_left = foundationScoresSelect_nonzero[['MAC_a_moral_nonmoral'] + [col for col in foundationScoresSelect_nonzero.columns if col.startswith('left')]]
foundationScoresSelect_nonzero_right = foundationScoresSelect_nonzero[['MAC_a_moral_nonmoral'] + [col for col in foundationScoresSelect_nonzero.columns if col.startswith('right')]]


In [ ]:
# correlations
correlation_matrix_left = foundationScoresSelect_nonzero_left.corr()
correlation_matrix_right = foundationScoresSelect_nonzero_right.corr()

In [ ]:
values_left = correlation_matrix_left['MAC_a_moral_nonmoral'][1:] 
values_right = correlation_matrix_right['MAC_a_moral_nonmoral'][1:] 

# Compute mean and SEM
mean_val_left = np.mean(values_left)
sem_val_left = np.std(values_left, ddof=1) / np.sqrt(len(values_left))

mean_val_right = np.mean(values_right)
sem_val_right = np.std(values_right, ddof=1) / np.sqrt(len(values_right))

# Create lists for plotting
means = [mean_val_left, mean_val_right]
sems = [sem_val_left, sem_val_right]
labels = ['Left TPJ', 'Right TPJ']

# Create the barplot
plt.figure(figsize=(5,6))
sns.barplot(x=labels, y=means, yerr=sems, color='skyblue', errorbar=None)

# Overlay individual dots
sns.stripplot(x=np.repeat(labels, [len(values_left), len(values_right)]),
              y=np.concatenate([values_left, values_right]),
              color='grey', size=6, jitter=True)

plt.ylabel('Correlation')
plt.title('Within-subject correlation of TPJ with MAC_a_moral_nonmoral (moral sentences only)')
plt.show()


In [ ]:

# One-sample t-test against 0
t_stat, p_value = stats.ttest_1samp(correlation_matrix_left['MAC_a_moral_nonmoral'][1:] , popmean=0)

print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
# One-sample t-test against 0
t_stat, p_value = stats.ttest_1samp(correlation_matrix_right['MAC_a_moral_nonmoral'][1:] , popmean=0)

print("t-statistic:", t_stat)
print("p-value:", p_value)